## Addition Interp
**"How do small transformers trained on addition data learn to add?"**

authored by : vorrjjard

In [ ]:
import torch as t
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

import transformer_lens

from transformer_lens import HookedTransformer, ActivationCache, HookedTransformerConfig
from transformer_lens.train import HookedTransformerTrainConfig, train

import typing
from typing import Literal, List

from dataclasses import dataclass

import numpy as np

import plotly.express as px

In [5]:
def pad(num: int, reversed: bool=False) -> str:
    num_str = str(num)
    while len(num_str) < 4:
        num_str = '0' + num_str
        
    if reversed:
            num_str = num_str[::-1]  

    return num_str

def generate_sample(cfg) -> list:
    max = cfg.max_output

    d1 = np.random.randint(cfg.min_output, cfg.max_output)
    d2 = np.random.randint(cfg.min_output, cfg.max_output - d1)

    sum = d1 + d2

    return f'{pad(d1)}+{pad(d2)}={pad(sum, reversed=True)}'

In [6]:
DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'
print(DEVICE)

cpu


In [7]:
@dataclass
class dataConfig:
    min_output: int = 0
    n_digits_input = 3
    n_digits_output = 4
    max_output: int = 5000 
    n_samples: int = 50000
data_cfg = dataConfig()
print(data_cfg)

dataConfig(min_output=0, max_output=5000, n_samples=50000)


In [8]:
vocab = {str(i): i for i in range(10)}
vocab['='] = 10
vocab['+'] = 11

In [9]:
vocab_inv = {i : str(i) for i in range(10)}
vocab_inv[10] = '='
vocab_inv[11] = '+'

Let's define a small vocabulary for our project. Hence, our `d_vocab = 10` 

In [10]:
# Generate samples, tokenize, and stack on top of a new batch dimension.

samples_tensor = t.stack(
    [t.tensor([vocab[char] for char in generate_sample(data_cfg)]) for n in range(0, data_cfg.n_samples)]
    ,dim=0)

In [11]:
class SumDataset(Dataset):
    def __init__(self : Dataset, token_dict : dict, samples):
        self.token_dict = token_dict
        self.samples = samples

    def __getitem__(self, idx):
        return {"tokens": self.samples[idx, :]}
    
    def __len__(self):
        return self.samples.shape[0]

In [12]:
ds = SumDataset(vocab, samples_tensor)
dl = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)

##### 2. Model Configuration / Training

In [13]:
def create_model(
    num_digits: int,
    seed: int,
    d_model: int,
    d_head: int,
    n_layers: int,
    n_heads: int,
    d_mlp: int | None,
    normalization_type: str | None,
    device: str = "cuda",
    **kwargs,  # ignore other kwargs
) -> HookedTransformer:
    t.manual_seed(seed)
    np.random.seed(seed)

    attn_only = d_mlp is None

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_ctx=num_digits * 3 + 2,  # We have [{a}, +, {b}, =, {a+b}]
        d_model=d_model,
        d_head=d_head,
        n_heads=n_heads,
        d_mlp=d_mlp,
        attn_only=attn_only,
        act_fn="relu",
        # We have all digits, and "+="
        d_vocab=12,
        # it's a small transformer so may as well use these hooks
        use_attn_result=True,
        use_split_qkv_input=True,
        use_hook_tokens=True,
        # Layernorm makes things way more accurate, even though it makes
        # mech interp a little more annoying!
        normalization_type=normalization_type,
        device=device,
    )

    model = HookedTransformer(cfg)
    return model

In [14]:
model = create_model(
    num_digits=4,
    seed=0,
    d_model=48,
    d_head=24,
    n_layers=2,
    n_heads=3,
    normalization_type="LN",
    d_mlp=None,
    device=DEVICE,
)

In [15]:
filename = '../models/sum_model.pt'

state_dict = t.load(filename, map_location=t.device('cpu'))

state_dict = model.center_writing_weights(t.load(filename, map_location=t.device('cpu')))
state_dict = model.center_unembed(state_dict)
state_dict = model.fold_layer_norm(state_dict)
state_dict = model.fold_value_biases(state_dict)
model.load_state_dict(state_dict, strict=False);

#### 3. Actual Interp

In [21]:
tokens = samples_tensor[3, :]
tokens

tensor([ 2,  7,  5,  2, 11,  0,  4,  7,  7, 10,  9,  2,  2,  3])

In [ ]:

token_labels = [vocab_inv[token.item()] for token in tokens]

logits, cache = model.run_with_cache(tokens.to(DEVICE))

In [17]:
attn_scores_0 = cache['blocks.0.attn.hook_pattern']
attn_scores_1 = cache['blocks.1.attn.hook_pattern']

attn_scores = t.cat([attn_scores_0, attn_scores_1], dim=0)

In [28]:
sum_scores = attn_scores[:, :, -5:-1, :]

pos_labels = [f"[{i}]{label}" for i, label in enumerate(token_labels)]

px.imshow(
    sum_scores.detach().cpu().numpy(),
    facet_col=0, #col is layers
    facet_row=1,
    labels=dict(x='Key', y='Query'),
    x=pos_labels,
    y=pos_labels[-5:-1],
    zmin=0,
    zmax=1,
    color_continuous_scale=[[0, 'black'], [1, '#fde725']],
)

It seems that the digits use digits from other positions to predict their output. Some observations : 
1. There seem to be some diagonal attention patterns which are outside the identity. I'm not sure what this means yet.